In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================
# 1. DQN MODEL + FINAL MODEL LOAD
# =============================================
class DQN(nn.Module):
    def __init__(self, state_size=7, action_size=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, action_size)
        )
    def forward(self, x):
        return self.net(x)

policy_net = DQN().to(device)
policy_net.load_state_dict(torch.load("../models/quantum_alpha_final.pth", map_location=device))
policy_net.eval()
print(" Final model loaded: quantum_alpha_final.pth")

# =============================================
# 2. ENVIRONMENT
# =============================================
class QuantumAlphaEnv:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.feature_cols = ["mom_20_norm", "vol_signal_norm", "trend_signal_norm",
                             "dd_signal_norm", "vix_signal_norm", "breadth_signal_norm"]
        self.max_steps = len(self.data) - 1
        self.reset()
   
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = 1.0
        return self._get_observation()
   
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        obs = row[self.feature_cols].values.astype(np.float32)
        return np.append(obs, self.position)
   
    def step(self, action):
        prev_position = self.position
        new_position = {0: 0, 1: 1, 2: -1}[action]
        ret = self.data.iloc[self.current_step]["nifty_ret"]
        cost = abs(new_position - prev_position) * 0.0005
        net_ret = prev_position * ret - cost
        self.balance *= (1 + net_ret)
        self.position = new_position
        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self._get_observation(), net_ret, done, {"net_ret": net_ret, "position": self.position}

# =============================================
# 3. WALK-FORWARD VALIDATION (Fixed with proper dates)
# =============================================
full_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
full_data = full_data.join(pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)[["nifty_ret"]])

windows = [
    ("2019-01-01", "2021-12-31", "2022-01-01", "2023-12-31"),
    ("2020-01-01", "2022-12-31", "2023-01-01", "2024-12-31"),
    ("2021-01-01", "2023-12-31", "2024-01-01", "2025-12-31")
]

results = []
for train_start, train_end, test_start, test_end in windows:
    train_data = full_data.loc[train_start:train_end]
    test_data = full_data.loc[test_start:test_end]
    
    env = QuantumAlphaEnv(test_data)
    state = env.reset()
    done = False
    equity = 1.0
    equity_curve = [1.0]
    positions = []
    
    while not done:
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            action = policy_net(s).argmax().item()
        next_state, _, done, info = env.step(action)
        equity *= (1 + info["net_ret"])
        equity_curve.append(equity)
        positions.append(info["position"])
        state = next_state
    
    equity_arr = np.array(equity_curve)
    returns = np.diff(equity_arr) / equity_arr[:-1]
    
    cagr = equity_arr[-1] ** (252 / len(returns)) - 1
    vol = np.std(returns) * np.sqrt(252)
    sharpe = (np.mean(returns) - 0.065/252) / (np.std(returns) + 1e-8) * np.sqrt(252)
    max_dd = (equity_arr / np.maximum.accumulate(equity_arr) - 1).min()
    
    results.append({
        "Train": f"{train_start}–{train_end}",
        "Test": f"{test_start}–{test_end}",
        "CAGR": round(cagr, 4),
        "Sharpe": round(sharpe, 4),
        "MaxDD": round(max_dd, 4),
        "Flat %": round((np.array(positions) == 0).mean() * 100, 1)
    })

wf_df = pd.DataFrame(results)
print("\n=== WALK-FORWARD VALIDATION RESULTS ===")
print(wf_df)

# =============================================
# 4. STRESS TEST PERIODS
# =============================================
stress_periods = {
    "COVID Crash (2020)": ("2020-02-01", "2020-04-30"),
    "2022 High Volatility": ("2022-01-01", "2022-10-31"),
    "Oil Shock 2022": ("2022-02-01", "2022-06-30")
}

stress_results = []
for name, (start, end) in stress_periods.items():
    stress_data = full_data.loc[start:end]
    env = QuantumAlphaEnv(stress_data)
    state = env.reset()
    done = False
    equity = 1.0
    equity_curve = [1.0]
    positions = []
    while not done:
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            action = policy_net(s).argmax().item()
        next_state, _, done, info = env.step(action)
        equity *= (1 + info["net_ret"])
        equity_curve.append(equity)
        positions.append(info["position"])
        state = next_state
    equity_arr = np.array(equity_curve)
    returns = np.diff(equity_arr) / equity_arr[:-1]
    cagr = equity_arr[-1] ** (252 / len(returns)) - 1
    sharpe = (np.mean(returns) - 0.065/252) / (np.std(returns) + 1e-8) * np.sqrt(252)
    max_dd = (equity_arr / np.maximum.accumulate(equity_arr) - 1).min()
    stress_results.append({
        "Period": name,
        "CAGR": round(cagr, 4),
        "Sharpe": round(sharpe, 4),
        "MaxDD": round(max_dd, 4),
        "Flat %": round((np.array(positions) == 0).mean() * 100, 1)
    })

stress_df = pd.DataFrame(stress_results)
print("\n=== STRESS TEST RESULTS ===")
print(stress_df)

 Final model loaded: quantum_alpha_final.pth

=== WALK-FORWARD VALIDATION RESULTS ===
                   Train                   Test    CAGR  Sharpe   MaxDD  \
0  2019-01-01–2021-12-31  2022-01-01–2023-12-31  0.1167  0.5595 -0.0614   
1  2020-01-01–2022-12-31  2023-01-01–2024-12-31 -0.0569 -2.0239 -0.1507   
2  2021-01-01–2023-12-31  2024-01-01–2025-12-31 -0.0330 -1.4763 -0.1069   

   Flat %  
0    71.3  
1    72.0  
2    63.5  

=== STRESS TEST RESULTS ===
                 Period    CAGR  Sharpe   MaxDD  Flat %
0    COVID Crash (2020)  1.2506  1.7177 -0.1378    33.3
1  2022 High Volatility  0.3390  1.9382 -0.0366    62.3
2        Oil Shock 2022  0.3558  1.6236 -0.0366    52.0
